# Joined Time Series DataFrame

This notebook loads precomputed time series parameter CSVs (TSS, precipitation, discharge, wind, and LULC),
merges them into two final dataframes (`df_local` and `df_regional`) and writes the merged CSV outputs.

Purpose:
- Explain columns and data sources.
- Produce consolidated tables for statistical analysis.

Data sources (CSV files):
- `area_TSS_time_series.csv` : TSS statistics per period.
- `chirps_time_series.csv` : precipitation (local and regional means).
- `discharge_time_series.csv` : river discharge time series.
- `era5_time_series.csv` : wind components (local and regional).
- `lulc_data.csv` : land-use / land-cover areas by watershed and class.

Outputs:
- `df_merged_local.csv` and `df_merged_regional.csv` in the `datasets/Parameters Time series/merged_df` folder.

How to use:
1. Adjust paths below if your repo is in a different location.
2. Run the notebook from top to bottom to recreate merged CSVs.

In [132]:
# Load standard libraries
import pandas as pd
import os

## Load datasets
The next cells read CSV files created earlier in the workflow. Each file contains time-series summaries by sampling period.
Make sure the `datasets/Parameters Time series/Results` folder exists and contains the expected CSVs.

In [133]:
# Load water period definitions generated from hydrological analysis
df_period_limits = pd.read_csv(r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Water Period Definitions\water_period_limits.csv').drop(columns=['Unnamed: 0','year',	'month'])

# Convert date columns to datetime format for proper temporal processing
df_period_limits['lim_next'] = pd.to_datetime(df_period_limits['lim_next'])
df_period_limits['lim_previous'] = pd.to_datetime(df_period_limits['lim_previous'])

# Sort by start date to ensure chronological order
df_period_limits = df_period_limits.sort_values(by='lim_next').reset_index(drop=True)

df_period_limits.columns = ['time_start', 'time_finish', 'water_period']
# Display last rows to verify data completeness
display(df_period_limits.tail())

,time_start,time_finish,water_period
223,2024-11-23,2025-03-09,R
224,2025-03-09,2025-06-09,HW
225,2025-06-09,2025-09-09,F
226,2025-09-09,2025-12-09,LW
227,2025-12-09,2026-01-01,R


In [134]:
# Read TSS area statistics and keep only the columns we need
area_tss = pd.read_csv(r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Results\area_TSS_time_series.csv')\
    .drop(columns=['band_count'])
area_tss = area_tss[['TSS_max', 'TSS_mean', 'TSS_min', 'TSS_stdDev', 'area_km2',
                      'time_finish', 'time_start', 'water_period']].copy()
area_tss['year'] = pd.to_datetime(area_tss['time_finish']).dt.year
area_tss#.describe()

,TSS_max,TSS_mean,TSS_min,TSS_stdDev,area_km2,time_finish,time_start,water_period,year
0,231.388351,48.812000,8.646607,27.405530,1043.404013,1985-04-09,1984-12-24,R,1985
1,230.723541,26.369015,7.906408,18.243428,1091.495113,1985-07-09,1985-04-09,HW,1985
2,233.251251,36.788005,8.202882,24.029145,885.768756,1985-09-23,1985-07-09,F,1985
3,234.163910,59.018131,10.828718,48.529756,981.618495,1985-12-16,1985-09-23,LW,1985
4,231.440628,20.104853,8.064194,22.183980,1131.200198,1986-08-23,1986-06-08,F,1986
...,...,...,...,...,...,...,...,...,...
143,234.002213,178.787246,8.711090,63.964413,561.882528,2024-11-23,2024-08-23,LW,2024
144,235.141968,126.397186,7.947257,86.057653,368.463271,2025-03-09,2024-11-23,R,2025
145,215.393143,21.700121,7.871317,13.777758,1303.141133,2025-06-09,2025-03-09,HW,2025
146,215.301117,20.144170,7.920572,11.239461,1433.663327,2025-09-09,2025-06-09,F,2025


In [135]:
# Read precipitation timeseries (CHIRPS) - contains local and regional means
precipitation = pd.read_csv(r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Results\chirps_time_series.csv').drop(columns=['Unnamed: 0'])
precipitation.columns

Index(['mean_loc_chirps', 'mean_reg_chirps', 'time_finish', 'time_start',
       'water_period'],
      dtype='object')

In [136]:
# Read river discharge time series
discharge = pd.read_csv(r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Results\discharge_time_series.csv').drop(columns=['Unnamed: 0'])
discharge.columns

Index(['water_period', 'time_start', 'time_finish', 'mean_discharge'], dtype='object')

In [137]:
# Read ERA5-derived wind components (u,v) with local and regional aggregates
wind = pd.read_csv(r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Results\era5_time_series.csv').drop(columns=['Unnamed: 0', 'year'])
wind.columns

Index(['time_finish', 'time_start', 'u_wind_loc', 'u_wind_reg', 'v_wind_loc',
       'v_wind_reg', 'water_period'],
      dtype='object')

In [138]:
# Read land-use / land-cover area summaries by watershed and class
lulc = pd.read_csv(r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Results\lulc_data.csv').drop(columns=['Unnamed: 0'])
lulc.columns

Index(['year', 'class_name', 'watershed', 'area_km2'], dtype='object')

# organize dataframes

## Local watershed

In [139]:
# Select local precipitation summary columns
precipitation_loc = precipitation[['mean_loc_chirps','time_finish', 'time_start', 'water_period',]].copy()
precipitation_loc.tail()

,mean_loc_chirps,time_finish,time_start,water_period
164,1205.653605,2025-03-09,2024-11-23,R
165,837.943684,2025-06-09,2025-03-09,HW
166,268.558520,2025-09-09,2025-06-09,F
167,218.504847,2025-12-09,2025-09-09,LW
168,90.892970,2026-01-01,2025-12-09,R


In [140]:
# Select local wind components
wind_loc = wind[['time_finish', 'time_start', 'u_wind_loc',
        'v_wind_loc', 'water_period',]].copy()
wind_loc.head()

,time_finish,time_start,u_wind_loc,v_wind_loc,water_period
0,1984-03-08,1983-11-23,-1.621382,-0.472935,R
1,1984-06-16,1984-03-08,-1.541157,-0.225848,HW
2,1984-09-16,1984-06-16,-1.736837,-0.122555,F
3,1984-12-24,1984-09-16,-1.498160,-0.092668,LW
4,1985-04-09,1984-12-24,-1.803958,-0.469250,R


In [141]:
# Ensure time columns are proper datetimes and have a 'year' column for merging
list = [df_period_limits,
        area_tss,
        precipitation_loc,
        discharge,
        wind_loc]

for df in list:
    df['time_finish'] = pd.to_datetime(df['time_finish'])
    df['time_start'] = pd.to_datetime(df['time_start'])
    df['year'] = df['time_finish'].dt.year
    df = df.sort_values('time_finish').reset_index(drop=True)
    print(df.columns)

Index(['time_start', 'time_finish', 'water_period', 'year'], dtype='object')
Index(['TSS_max', 'TSS_mean', 'TSS_min', 'TSS_stdDev', 'area_km2',
       'time_finish', 'time_start', 'water_period', 'year'],
      dtype='object')
Index(['mean_loc_chirps', 'time_finish', 'time_start', 'water_period', 'year'], dtype='object')
Index(['water_period', 'time_start', 'time_finish', 'mean_discharge', 'year'], dtype='object')
Index(['time_finish', 'time_start', 'u_wind_loc', 'v_wind_loc', 'water_period',
       'year'],
      dtype='object')


In [142]:
# Merge datasets to create a base dataframe, then add local precipitation and wind
df_base = area_tss.merge(df_period_limits,on=['time_finish', 'time_start', 'water_period','year'],how='outer')
df_base = df_base.merge(discharge, on=['time_finish', 'time_start', 'water_period','year'],how='outer')
df_local = df_base.merge(precipitation_loc, on=['time_finish', 'time_start', 'water_period','year'],how='outer')
df_local = df_local.merge(wind_loc, on=['time_finish', 'time_start', 'water_period','year'],how='outer')
df_local

,TSS_max,TSS_mean,TSS_min,TSS_stdDev,area_km2,time_finish,time_start,water_period,year,mean_discharge,mean_loc_chirps,u_wind_loc,v_wind_loc
0,NaN,NaN,NaN,NaN,NaN,1968-09-30 12:00:00,1968-06-30 12:00:00,HW,1968,158708.786522,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,1969-08-01 12:00:00,1968-09-30 12:00:00,F,1969,120805.471815,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,1970-02-07 18:00:00,1969-08-01 12:00:00,LW,1970,114822.916947,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,1970-04-24 06:00:00,1970-02-07 18:00:00,R,1970,170514.484868,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,1970-07-16 18:00:00,1970-04-24 06:00:00,HW,1970,212042.247831,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
344,235.141968,126.397186,7.947257,86.057653,368.463271,2025-03-09 00:00:00,2024-11-23 00:00:00,R,2025,124708.110243,1205.653605,-1.683873,-0.251720
345,215.393143,21.700121,7.871317,13.777758,1303.141133,2025-06-09 00:00:00,2025-03-09 00:00:00,HW,2025,223941.292151,837.943684,-2.227315,-0.343642
346,215.301117,20.144170,7.920572,11.239461,1433.663327,2025-09-09 00:00:00,2025-06-09 00:00:00,F,2025,231944.084717,268.558520,-2.356343,-0.204407
347,231.284851,58.669482,8.254883,27.916471,1112.831699,2025-12-09 00:00:00,2025-09-09 00:00:00,LW,2025,NaN,218.504847,-2.075493,0.143591


## Regional watershed

In [143]:
# Select regional precipitation summary columns
precipitation_reg = precipitation[['mean_reg_chirps','time_finish', 'time_start', 'water_period']].copy()

precipitation_reg.head()

,mean_reg_chirps,time_finish,time_start,water_period
0,580.944981,1984-03-08,1983-11-23,R
1,974.719542,1984-06-16,1984-03-08,HW
2,536.004241,1984-09-16,1984-06-16,F
3,408.400765,1984-12-24,1984-09-16,LW
4,753.868485,1985-04-09,1984-12-24,R


In [144]:
# Select regional wind components
wind_reg = wind[['time_finish', 'time_start', 'u_wind_reg',
        'v_wind_reg', 'water_period']].copy()
wind_reg.head()

,time_finish,time_start,u_wind_reg,v_wind_reg,water_period
0,1984-03-08,1983-11-23,-0.795716,-0.475848,R
1,1984-06-16,1984-03-08,-0.651839,-0.285321,HW
2,1984-09-16,1984-06-16,-0.575338,-0.221219,F
3,1984-12-24,1984-09-16,-0.634260,-0.284136,LW
4,1985-04-09,1984-12-24,-0.945100,-0.693335,R


In [145]:
# Ensure time columns are proper datetimes and have a 'year' column for merging
list = [precipitation_reg,
        wind_reg]

for df in list:
    df['time_finish'] = pd.to_datetime(df['time_finish'])
    df['time_start'] = pd.to_datetime(df['time_start'])
    df['year'] = df['time_finish'].dt.year
    df = df.sort_values('time_finish').reset_index(drop=True)
    print(df.columns)

Index(['mean_reg_chirps', 'time_finish', 'time_start', 'water_period', 'year'], dtype='object')
Index(['time_finish', 'time_start', 'u_wind_reg', 'v_wind_reg', 'water_period',
       'year'],
      dtype='object')


In [146]:
# Build the regional dataframe by adding regional precipitation and wind
df_regional = df_base.merge(precipitation_reg, on=['time_finish', 'time_start', 'water_period','year'],how='outer')
df_regional = df_regional.merge(wind_reg, on=['time_finish', 'time_start', 'water_period','year'],how='outer')
df_regional

,TSS_max,TSS_mean,TSS_min,TSS_stdDev,area_km2,time_finish,time_start,water_period,year,mean_discharge,mean_reg_chirps,u_wind_reg,v_wind_reg
0,NaN,NaN,NaN,NaN,NaN,1968-09-30 12:00:00,1968-06-30 12:00:00,HW,1968,158708.786522,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,1969-08-01 12:00:00,1968-09-30 12:00:00,F,1969,120805.471815,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,1970-02-07 18:00:00,1969-08-01 12:00:00,LW,1970,114822.916947,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,1970-04-24 06:00:00,1970-02-07 18:00:00,R,1970,170514.484868,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,1970-07-16 18:00:00,1970-04-24 06:00:00,HW,1970,212042.247831,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
344,235.141968,126.397186,7.947257,86.057653,368.463271,2025-03-09 00:00:00,2024-11-23 00:00:00,R,2025,124708.110243,898.932983,-0.812905,-0.467850
345,215.393143,21.700121,7.871317,13.777758,1303.141133,2025-06-09 00:00:00,2025-03-09 00:00:00,HW,2025,223941.292151,856.490467,-0.830971,-0.289036
346,215.301117,20.144170,7.920572,11.239461,1433.663327,2025-09-09 00:00:00,2025-06-09 00:00:00,F,2025,231944.084717,382.916565,-0.814023,-0.274272
347,231.284851,58.669482,8.254883,27.916471,1112.831699,2025-12-09 00:00:00,2025-09-09 00:00:00,LW,2025,NaN,311.053055,-0.731969,-0.141838


# include lulc data

## Local Watershed

In [147]:
# Extract LULC for the Local Watershed and split natural vs anthropogenic classes
lulc_local = lulc.loc[lulc['watershed'] == 'Local Watershed'].copy()
lulc_local_nat = lulc_local.loc[lulc_local['class_name'] == 'Natural'].copy()
lulc_local_nat = lulc_local_nat.rename(columns={'area_km2': 'natural_km2'}).drop(columns=['class_name'])
lulc_local_nat.head()

,year,watershed,natural_km2
2,1985,Local Watershed,1838.000682
6,1986,Local Watershed,1865.902793
10,1987,Local Watershed,1809.635155
14,1988,Local Watershed,1759.578980
18,1989,Local Watershed,1691.760531


In [148]:
# Prepare anthropogenic area column for the local watershed
lulc_local_ant = lulc_local.loc[lulc_local['class_name'] != 'Natural'].copy()
lulc_local_ant = lulc_local_ant.rename(columns={'area_km2': 'anthropogenic_km2'}).drop(columns=['class_name'])
lulc_local_ant.head()

,year,watershed,anthropogenic_km2
0,1985,Local Watershed,48.897800
4,1986,Local Watershed,49.877077
8,1987,Local Watershed,91.981528
12,1988,Local Watershed,93.984196
16,1989,Local Watershed,69.071330


In [149]:
lulc_local = lulc_local_ant.merge(lulc_local_nat, on=['year','watershed'], how='outer')
lulc_local.head()

,year,watershed,anthropogenic_km2,natural_km2
0,1985,Local Watershed,48.897800,1838.000682
1,1986,Local Watershed,49.877077,1865.902793
2,1987,Local Watershed,91.981528,1809.635155
3,1988,Local Watershed,93.984196,1759.578980
4,1989,Local Watershed,69.071330,1691.760531


In [150]:
lulc_local.tail()

,year,watershed,anthropogenic_km2,natural_km2
35,2020,Local Watershed,202.905644,1559.941249
36,2021,Local Watershed,196.595229,1521.152454
37,2022,Local Watershed,195.541853,1451.598098
38,2023,Local Watershed,199.570460,1626.906567
39,2024,Local Watershed,224.678151,1674.648490


In [151]:
# Attach LULC summaries to the local dataframe and add a watershed label
df_local = df_local.merge(lulc_local, on ="year", how='left')
df_local['watershed'] = 'Local Watershed'
df_local.tail(50)

,TSS_max,TSS_mean,TSS_min,TSS_stdDev,area_km2,time_finish,time_start,water_period,year,mean_discharge,mean_loc_chirps,u_wind_loc,v_wind_loc,watershed,anthropogenic_km2,natural_km2
299,NaN,NaN,NaN,NaN,NaN,2018-12-24 00:00:00,2018-09-23 18:00:00,LW,2018,111887.203043,NaN,NaN,NaN,Local Watershed,221.829328,1548.692857
300,231.576080,50.536509,8.171493,23.514904,1247.324734,2019-04-09 00:00:00,2018-12-24 00:00:00,R,2019,186162.615607,998.416508,-2.102817,-0.538821,Local Watershed,214.503149,1514.696563
301,220.105042,18.846145,7.871317,12.729591,1399.770388,2019-07-09 00:00:00,2019-04-09 00:00:00,HW,2019,NaN,669.469607,-2.221584,-0.253069,Local Watershed,214.503149,1514.696563
302,NaN,NaN,NaN,NaN,NaN,2019-07-09 06:00:00,2019-04-09 00:00:00,HW,2019,237289.610543,NaN,NaN,NaN,Local Watershed,214.503149,1514.696563
303,220.707840,18.186829,7.932969,8.625178,1471.418564,2019-09-23 00:00:00,2019-07-09 00:00:00,F,2019,NaN,80.082342,-2.235029,-0.270787,Local Watershed,214.503149,1514.696563
304,NaN,NaN,NaN,NaN,NaN,2019-09-23 18:00:00,2019-07-09 06:00:00,F,2019,203887.674737,NaN,NaN,NaN,Local Watershed,214.503149,1514.696563
305,234.160919,84.635910,9.532992,51.145077,951.693159,2019-12-24 00:00:00,2019-09-23 00:00:00,LW,2019,NaN,397.417101,-1.579192,0.123943,Local Watershed,214.503149,1514.696563
306,NaN,NaN,NaN,NaN,NaN,2019-12-24 06:00:00,2019-09-23 18:00:00,LW,2019,118425.079457,NaN,NaN,NaN,Local Watershed,214.503149,1514.696563
307,229.115036,44.412756,7.932969,16.887689,1322.433629,2020-04-08 00:00:00,2019-12-24 00:00:00,R,2020,NaN,933.026617,-1.600541,-0.541310,Local Watershed,202.905644,1559.941249
308,NaN,NaN,NaN,NaN,NaN,2020-04-08 18:00:00,2019-12-24 06:00:00,R,2020,177562.878019,NaN,NaN,NaN,Local Watershed,202.905644,1559.941249


## Regional Watershed

In [152]:
lulc_regional = lulc.loc[lulc['watershed'] != 'Local Watershed'].copy()
lulc_regional_nat = lulc_regional.loc[lulc_regional['class_name'] == 'Natural'].copy()
lulc_regional_nat = lulc_regional_nat.rename(columns={'area_km2': 'natural_km2'}).drop(columns=['class_name'])
lulc_regional_nat.head()

,year,watershed,natural_km2
3,1985,Regional Watershed,305485.474607
7,1986,Regional Watershed,305474.722955
11,1987,Regional Watershed,304938.202241
15,1988,Regional Watershed,304585.406387
19,1989,Regional Watershed,304367.353637


In [153]:
lulc_regional_ant = lulc_regional.loc[lulc_regional['class_name'] != 'Natural'].copy()
lulc_regional_ant = lulc_regional_ant.rename(columns={'area_km2': 'anthropogenic_km2'}).drop(columns=['class_name'])
lulc_regional_ant.head()

,year,watershed,anthropogenic_km2
1,1985,Regional Watershed,1637.268807
5,1986,Regional Watershed,1567.067862
9,1987,Regional Watershed,1769.637364
13,1988,Regional Watershed,1819.227134
17,1989,Regional Watershed,1736.977802


In [154]:
lulc_regional = lulc_regional_ant.merge(lulc_regional_nat, on=['year','watershed'], how='outer')
lulc_regional.head()

,year,watershed,anthropogenic_km2,natural_km2
0,1985,Regional Watershed,1637.268807,305485.474607
1,1986,Regional Watershed,1567.067862,305474.722955
2,1987,Regional Watershed,1769.637364,304938.202241
3,1988,Regional Watershed,1819.227134,304585.406387
4,1989,Regional Watershed,1736.977802,304367.353637


In [155]:
# Attach LULC summaries to the regional dataframe and add a watershed label
df_regional = df_regional.merge(lulc_regional, on ="year", how='left')
df_regional['watershed'] = 'Regional Watershed'
df_regional.tail()

,TSS_max,TSS_mean,TSS_min,TSS_stdDev,area_km2,time_finish,time_start,water_period,year,mean_discharge,mean_reg_chirps,u_wind_reg,v_wind_reg,watershed,anthropogenic_km2,natural_km2
344,235.141968,126.397186,7.947257,86.057653,368.463271,2025-03-09,2024-11-23,R,2025,124708.110243,898.932983,-0.812905,-0.467850,Regional Watershed,NaN,NaN
345,215.393143,21.700121,7.871317,13.777758,1303.141133,2025-06-09,2025-03-09,HW,2025,223941.292151,856.490467,-0.830971,-0.289036,Regional Watershed,NaN,NaN
346,215.301117,20.144170,7.920572,11.239461,1433.663327,2025-09-09,2025-06-09,F,2025,231944.084717,382.916565,-0.814023,-0.274272,Regional Watershed,NaN,NaN
347,231.284851,58.669482,8.254883,27.916471,1112.831699,2025-12-09,2025-09-09,LW,2025,NaN,311.053055,-0.731969,-0.141838,Regional Watershed,NaN,NaN
348,NaN,NaN,NaN,NaN,NaN,2026-01-01,2025-12-09,R,2026,NaN,136.640289,NaN,NaN,Regional Watershed,NaN,NaN


# Complete data

In [156]:
df_local['year'] = df_local['year'].replace({2026:2025})
df_regional['year'] = df_regional['year'].replace({2026:2025})

In [157]:
df_local = df_local.rename(columns={'mean_loc_chirps': "precipitation", "u_wind_loc":"u_wind", "v_wind_loc":"v_wind"})
df_regional = df_regional.rename(columns={'mean_reg_chirps': "precipitation", "u_wind_reg":"u_wind", "v_wind_reg":"v_wind"})

In [158]:
df_local = df_local.loc[df_local['time_finish'] >= "1985-01-01"].copy()
df_regional = df_regional.loc[df_regional['time_finish'] >= "1985-01-01"].copy()

In [159]:
df_regional.set_index('time_finish', inplace=True)
df_local.set_index('time_finish', inplace=True)

In [160]:
list = [df_local, df_regional]
for df_ in list:
    for column in df_.columns:
        print(f"Column: {column}, Missing values: {df_[column].isna().sum()}")
        if column in ['anthropogenic_km2', 'natural_km2']:
            print(f"Interpolating missing values in column: {column} with Time-Aware Interpolation")
            df_[column] = df_[column].interpolate(method='time', limit=5)
        elif column in ['mean_discharge','precipitation','u_wind','v_wind']:
            print(f"Interpolating missing values in column: {column} with Polynomial Interpolation (2nd order)")
            df_[column] = df_[column].interpolate(method='polynomial', order=3, limit=6)
        elif column in ['TSS_max','TSS_mean','TSS_min','TSS_stdDev','area_km2']:
            print(f"Interpolating missing values in column: {column + '_fill'} with Polynomial Interpolation (2nd order)")
            df_[column + '_fill'] = df_[column].interpolate(method='polynomial', order=3, limit=6)
        else:
            print("No interpolation applied for this column.")
    

Column: TSS_max, Missing values: 136
Interpolating missing values in column: TSS_max_fill with Polynomial Interpolation (2nd order)
Column: TSS_mean, Missing values: 136
Interpolating missing values in column: TSS_mean_fill with Polynomial Interpolation (2nd order)
Column: TSS_min, Missing values: 136
Interpolating missing values in column: TSS_min_fill with Polynomial Interpolation (2nd order)
Column: TSS_stdDev, Missing values: 136
Interpolating missing values in column: TSS_stdDev_fill with Polynomial Interpolation (2nd order)
Column: area_km2, Missing values: 136
Interpolating missing values in column: area_km2_fill with Polynomial Interpolation (2nd order)
Column: time_start, Missing values: 0
No interpolation applied for this column.
Column: water_period, Missing values: 0
No interpolation applied for this column.
Column: year, Missing values: 0
No interpolation applied for this column.
Column: mean_discharge, Missing values: 121
Interpolating missing values in column: mean_disch

In [161]:
# Export merged dataframes to CSV for downstream analysis
exp_dir = r"C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\merged_df"
df_local_fill = df_local.drop(columns=['area_km2','TSS_mean']).rename(columns={"TSS_mean_fill": "TSS_mean","area_km2_fill": "area_km2"})
df_local_fill.to_csv(os.path.join(exp_dir,'df_fillTSS_local.csv'))
df_regional_fill = df_regional.drop(columns=['area_km2','TSS_mean']).rename(columns={"TSS_mean_fill": "TSS_mean","area_km2_fill": "area_km2"})
df_regional.to_csv(os.path.join(exp_dir,'df_fillTSS_regional.csv'))

In [162]:
df_local = df_local.dropna().drop(columns=['area_km2_fill','TSS_mean_fill'])
df_regional = df_regional.dropna().drop(columns=['area_km2_fill','TSS_mean_fill'])

In [163]:
df_local

,TSS_max,TSS_mean,TSS_min,TSS_stdDev,area_km2,time_start,water_period,year,mean_discharge,precipitation,u_wind,v_wind,watershed,anthropogenic_km2,natural_km2,TSS_max_fill,TSS_min_fill,TSS_stdDev_fill
time_finish,,,,,,,,,,,,,,,,,,
1985-04-09,231.388351,48.812000,8.646607,27.405530,1043.404013,1984-12-24,R,1985,166387.875673,1082.939016,-1.803958,-0.469250,Local Watershed,48.897800,1838.000682,231.388351,8.646607,27.405530
1985-07-09,230.723541,26.369015,7.906408,18.243428,1091.495113,1985-04-09,HW,1985,191041.161477,578.634586,-2.042476,-0.087150,Local Watershed,48.897800,1838.000682,230.723541,7.906408,18.243428
1985-09-23,233.251251,36.788005,8.202882,24.029145,885.768756,1985-07-09,F,1985,169404.971454,125.430803,-1.711139,-0.183250,Local Watershed,48.897800,1838.000682,233.251251,8.202882,24.029145
1985-12-16,234.163910,59.018131,10.828718,48.529756,981.618495,1985-09-23,LW,1985,119094.384372,411.507501,-1.364044,-0.094174,Local Watershed,48.897800,1838.000682,234.163910,10.828718,48.529756
1986-08-23,231.440628,20.104853,8.064194,22.183980,1131.200198,1986-06-08,F,1986,212255.269086,102.247481,-2.171907,0.159701,Local Watershed,49.877077,1865.902793,231.440628,8.064194,22.183980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-23,220.766983,33.658594,7.932969,12.795185,1310.040552,2024-06-08,F,2024,183466.799221,104.677317,-2.373738,-0.348760,Local Watershed,224.678151,1674.648490,220.766983,7.932969,12.795185
2024-11-23,234.002213,178.787246,8.711090,63.964413,561.882528,2024-08-23,LW,2024,74561.514826,95.168296,-2.496754,-0.012355,Local Watershed,224.678151,1674.648490,234.002213,8.711090,63.964413
2025-03-09,235.141968,126.397186,7.947257,86.057653,368.463271,2024-11-23,R,2025,124708.110243,1205.653605,-1.683873,-0.251720,Local Watershed,224.678151,1674.648490,235.141968,7.947257,86.057653


In [164]:
df_local.to_csv(os.path.join(exp_dir,'df_merged_local.csv'))
df_regional.to_csv(os.path.join(exp_dir,'df_merged_regional.csv'))